# Instalando pacotes

In [1]:
from io import BytesIO
import sys
!{sys.executable} -m pip install fsspec s3fs oci ocifs 
!{sys.executable} -m pip install pandas numpy==1.24.3
!{sys.executable} -m pip uninstall pyarrow -y
!{sys.executable} -m pip install pyarrow==12.0.1

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Found existing installation: pyarrow 12.0.1
Uninstalling pyarrow-12.0.1:
  Successfully uninstalled pyarrow-12.0.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-12.0.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (39.0 MB)


# Carregando pacotes

In [2]:
import oci
import ocifs
import pandas as pd
import sys
import os
import pyarrow
# Funcoes customizadas
import configs.function_basic as funcoes

# Conexão ao repositório via OCI

In [3]:
#Buckets e nomes de saída nuvem = "oci://"
namespace = "@grxzqsiaote6/"
pasta_in = 'Feature_store/book_variaveis_03.parquet'
pasta_in_trusted = 'bases_recarga/'
pasta_out = 'Feature_store/'

bucket_book = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_in}" 
bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in_trusted}"
bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

## Carregando databases

#### Book_03

In [4]:
from ocifs import OCIFileSystem

fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_03.parquet")
files = fs.ls(bucket_book)

print(files)

['BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_03.parquet']


In [5]:
# Carregando book_03
df_book_03 = pd.read_parquet(
    bucket_book, #"oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_03.parquet",
    storage_options={"config": "~/.oci/config"})

print("Book 03 data shape:", df_book_03.shape)

Book 03 data shape: (3734429, 108)


In [6]:
df_book_03.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3734429 entries, 0 to 3734428
Columns: 108 entries, Ano to ts_proc_partition
dtypes: Int32(3), Int64(9), Int8(1), boolean(1), category(1), datetime64[ns](2), float32(74), float64(4), int32(2), object(10), string(1)
memory usage: 1.9+ GB


#### Base Recarga

In [7]:
fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_03.parquet")
files = fs.ls(bucket_trusted)

print(files)

['TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2023-10-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2023-11-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2023-12-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-01-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-02-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-03-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-04-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-05-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-06-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-07-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-08-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-09-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-10-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-11-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2024-12-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2025-01-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2025-02-01', 'TRUSTED@grxzqsiaote6/bases_recarga/SAFRA=2025-03-01', 'TRUSTED@

In [ ]:
## Carregando todos arquivos em parquet de uma pasta
df_dados_recarga = pd.read_parquet(
    bucket_trusted, #"oci://TRUSTED@grxzqsiaote6/bases_recarga/",
    storage_options={"config": "~/.oci/config"})
print('Base Dados Recarga data shape:', df_dados_recarga.shape)

In [ ]:
df_dados_recarga.info()

#### Ajuste no Dataset de Recargas

Para diminuir o tamanho da base processada e focarmos no nosso problema de negócio, iremos filtrar apenas os CPFs que estão na base do `df_book_03`

Também iremos criar a coluna `SAFRA` a partir da coluna `DAT_INSERCAO_CREDITO`

In [ ]:
selecao_publico = df_book_03['NUM_CPF'].drop_duplicates()

# Selecionando na base de recarga apenas os CPFs presentes na base de score bureau movel
df_base_recarga_selecionada = df_dados_recarga[df_dados_recarga['NUM_CPF'].isin(selecao_publico)]
df_base_recarga_selecionada.head()

In [ ]:
# Coonferindo se temos valores nulos em DAT_INSERCAO_CREDITO
df_base_recarga_selecionada ['DAT_INSERCAO_CREDITO'].isnull().sum()

In [ ]:
# Criando coluna SAFRA
df_base_recarga_selecionada = funcoes.criar_coluna_safra(df_base_recarga_selecionada, 'DAT_INSERCAO_CREDITO')

In [ ]:
df_base_recarga_selecionada['SAFRA'].value_counts(dropna=False)

In [ ]:
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] \
    .value_counts(normalize=True) \
    .mul(100) \
    .round(2) \
    .rename('percentual_total')


In [ ]:
# Substituindo os valores diferentes de PREPG E AUTOC por OUTROS
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] = (
    df_base_recarga_selecionada['COD_PLATAFORMA_ATU']
        .where(
            df_base_recarga_selecionada['COD_PLATAFORMA_ATU'].isin(['PREPG', 'AUTOC']),
            'OUTROS'
        )
)

In [ ]:
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] \
    .value_counts(normalize=True) \
    .mul(100) \
    .round(2) \
    .rename('percentual_total')


In [ ]:
df_base_recarga_selecionada = df_base_recarga_selecionada[df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] == 'PREPG']

#### Ajuste dos Tipos de Dados

In [ ]:
# Ajuste dos tipos de dados para agregacao
df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'] = pd.to_numeric(df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_BONUS'] = pd.to_numeric(df_base_recarga_selecionada['VAL_BONUS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_REAL'] = pd.to_numeric(df_base_recarga_selecionada['VAL_REAL'], errors='coerce').fillna(0)
df_base_recarga_selecionada['FLAG_SOS'] = pd.to_numeric(df_base_recarga_selecionada['FLAG_SOS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VALOR_SOS'] = pd.to_numeric(df_base_recarga_selecionada['VALOR_SOS'], errors='coerce').fillna(0)

In [ ]:
df_base_recarga_selecionada

#### Criação das visões agregadas por SAFRA e CPF

In [ ]:
df_base_recarga_selecionada_agg = df_base_recarga_selecionada.groupby(['NUM_CPF', 'SAFRA']).agg(
        QTDE_RECARGAS=('NUM_CPF', 'size'),
        QTDE_NUMEROS=('DW_NUM_NTC', 'nunique'),
        VAL_CREDITO_INSERIDO=('VAL_CREDITO_INSERIDO', 'sum'),
        VAL_BONUS=('VAL_BONUS', 'sum'),
        VAL_REAL=('VAL_REAL', 'sum'),
        QTD_SOS=('FLAG_SOS', 'sum'),
        VALOR_SOS=('VALOR_SOS', 'sum'),
        
    ).reset_index()

In [ ]:
df_base_recarga_selecionada_agg

## Construção do Book Comportamental por SAFRA (Modelo de Crédito)

Este processo tem como objetivo a construção de um **book comportamental temporal** para modelagem de crédito, alinhado à predição de **FPD (First Payment Default)**.

### Visão Geral
Os dados comportamentais são inicialmente agregados no nível **CPF + SAFRA**, representando o comportamento observado em cada período mensal. A partir dessa base agregada, são criadas features históricas que capturam o comportamento passado do cliente em relação à **safra de referência**.

A base final do modelo é obtida por meio de um **LEFT JOIN** entre:
- **Base alvo (label)**: CPF + SAFRA com indicador FPD (0/1)
- **Base comportamental enriquecida**: histórico anterior ao mês da SAFRA

### Construção das Features Temporais
Para cada CPF, os dados são ordenados cronologicamente por SAFRA e são criadas defasagens temporais (*lags*) utilizando apenas informações do passado:

- Safra imediatamente anterior (t-1)
- Acumulado das últimas 3 safras (t-1 a t-3)
- Acumulado das últimas 6 safras (t-1 a t-6)

As defasagens são geradas via `groupby(CPF)` com `shift`, garantindo que **nenhuma informação da própria safra ou futura seja utilizada**, evitando vazamento temporal (*data leakage*).

### Robustez e Tratamento de Casos Especiais
- CPFs sem histórico anterior permanecem na base (LEFT JOIN), com valores nulos ou zerados.
- A ausência de histórico é considerada informação relevante para o modelo.
- O método é robusto a safras faltantes (meses sem registro), utilizando apenas o histórico efetivamente disponível.

### Garantias do Processo
- As features refletem exclusivamente o comportamento conhecido **até o momento da decisão de crédito**.
- A estrutura é reproduzível, auditável e adequada para uso em modelos supervisionados.
- O book final está preparado para técnicas de modelagem estatística e de machine learning.

Este desenho segue práticas consolidadas de modelagem de risco de crédito e permite expansão futura com métricas adicionais (médias, tendências, taxas e flags de histórico).


In [ ]:
variaveis = [
    'QTDE_RECARGAS',
    'QTDE_NUMEROS',
    'VAL_CREDITO_INSERIDO',
    'VAL_BONUS',
    'VAL_REAL',
    'QTD_SOS',
    'VALOR_SOS'
]

df_book_04 = funcoes.criar_lags_por_safra(
    df=df_base_recarga_selecionada_agg,
    col_cpf='NUM_CPF',
    col_safra='SAFRA',
    variaveis=variaveis,
    janelas=[1, 3, 6]
)

In [ ]:
df_book_04

In [ ]:
book_variaveis_04 = df_book_04.copy()

In [ ]:
book_variaveis_04 = pd.merge(
    book_03,
    df_book_04,
    on=['NUM_CPF', 'SAFRA'],
    how='left'
)

In [ ]:
book_variaveis_04

In [ ]:
book_variaveis_04['REGIAO_POSTAL_TXT'].value_counts()

In [ ]:
# Sanity check
df_book_03.shape[0] == df_book_04.shape[0]

In [ ]:
# Salvando o dataframe em parquet
book_variaveis_04.to_parquet(
    f"{bucket_feature_store}book_variaveis_04.parquet",
    engine="pyarrow",
    compression="snappy",
#    partition_cols=["SAFRA"],
    index=False,
    storage_options={"config": "~/.oci/config"}
)